# Convert Parquet to Arrow with chDB

In-process ClickHouse SQL, no server. Run `./generate.sh` first to create `data/events.parquet`.

Requires: `pip install chdb`

In [ ]:
import os
import chdb

os.chdir("data")

## Convert Parquet -> Arrow IPC

Same `SELECT ... FORMAT Arrow` as the CLI, written to a file.

In [ ]:
arrow_bytes = chdb.query("SELECT * FROM file('events.parquet') FORMAT Arrow").bytes()
with open("events_chdb.arrow", "wb") as f:
    f.write(arrow_bytes)
print(f"wrote events_chdb.arrow ({len(arrow_bytes)} bytes)")

## Confirm the schema + rows survived the round trip

In [ ]:
print(chdb.query("DESCRIBE file('events_chdb.arrow')", "CSV"))
print(chdb.query("SELECT * FROM file('events_chdb.arrow') LIMIT 5", "CSV"))

## Arrow is the zero-copy handoff to pandas/pyarrow

In [ ]:
df = chdb.query("SELECT * FROM file('events_chdb.arrow')", "DataFrame")
print(f"{df.shape[0]} rows x {df.shape[1]} cols")
print(df.dtypes)